In [1]:
# Setup
import os
from dotenv import load_dotenv
from google.cloud import bigquery
from anomaly_detector import diagnose, get_llm, suggest_fixes

load_dotenv()
PROJECT_ID = os.environ["GCP_PROJECT_ID"]
client = bigquery.Client(project=PROJECT_ID)
llm = get_llm()

In [2]:
# Dataset 1 — NYC Taxi Trips
df_nyc = client.query(
    "SELECT * FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022` LIMIT 1000"
).to_dataframe()
anomalies, diagnosis = diagnose(df_nyc, "NYC Taxi Trips 2022", llm)
fixes = suggest_fixes(anomalies, llm)
print("=== Anomalies ===")
print(anomalies)
print("\n=== Diagnosis ===")
print(diagnosis)
print("\n=== Suggested Fixes ===")
print(fixes)

c:\Users\Valer\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-04-03 01:21:00,659 INFO Starting diagnosis for dataset: NYC Taxi Trips 2022
2026-04-03 01:21:00,677 WARNING Null anomaly: passenger_count — 156 nulls (15.6%)
2026-04-03 01:21:00,678 WARNING Null anomaly: rate_code — 156 nulls (15.6%)
2026-04-03 01:21:00,678 WARNING Null anomaly: store_and_fwd_flag — 156 nulls (15.6%)
2026-04-03 01:21:00,678 WARNING Null anomaly: airport_fee — 156 nulls (15.6%)
2026-04-03 01:21:00,679 WARNING Zero value anomaly: passenger_count — 21 zeros
2026-04-03 01:21:12,893 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-03 01:21:12,904 INFO Diagnosis complete for dataset: NYC Taxi Trips 2022
2026-04-03 01:21:16,098 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
20

=== Anomalies ===
- passenger_count: 156 nulls (15.6%)
- rate_code: 156 nulls (15.6%)
- store_and_fwd_flag: 156 nulls (15.6%)
- airport_fee: 156 nulls (15.6%)
- passenger_count: 21 zero values

=== Diagnosis ===
### Anomaly 1: Null Values in `passenger_count`, `rate_code`, `store_and_fwd_flag`, and `airport_fee`
**Likely Cause**: These null values can arise from data entry errors, incomplete records at the time of trip completion, or issues during data ingestion from the source system.

**Pipeline Recommendation**:
1. **Data Imputation**: Consider replacing null values with a default value. For `passenger_count`, replacing nulls with the median or mode of the column is appropriate. 
   - For `rate_code`, use a default (e.g., 1 for "Standard rate") if a reasonable assumption can be made based on the trip characteristics.
   - For `store_and_fwd_flag`, defaulting to 'N' (not stored and forwarded) could be reasonable assuming most trips follow this condition.
   - For `airport_fee`, if ap

In [3]:
# Dataset 2 — Chicago Taxi Trips
df_chicago = client.query(
    "SELECT * FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips` LIMIT 1000"
).to_dataframe()
anomalies, diagnosis = diagnose(df_chicago, "Chicago Taxi Trips", llm)
fixes = suggest_fixes(anomalies, llm)
print("=== Anomalies ===")
print(anomalies)
print("\n=== Diagnosis ===")
print(diagnosis)
print("\n=== Suggested Fixes ===")
print(fixes)

c:\Users\Valer\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-04-03 01:21:17,420 INFO Starting diagnosis for dataset: Chicago Taxi Trips
2026-04-03 01:21:17,422 WARNING Null anomaly: pickup_census_tract — 996 nulls (99.6%)
2026-04-03 01:21:17,423 WARNING Null anomaly: dropoff_census_tract — 996 nulls (99.6%)
2026-04-03 01:21:17,423 WARNING Null anomaly: pickup_community_area — 1000 nulls (100.0%)
2026-04-03 01:21:17,424 WARNING Null anomaly: dropoff_community_area — 1000 nulls (100.0%)
2026-04-03 01:21:17,424 WARNING Null anomaly: tolls — 32 nulls (3.2%)
2026-04-03 01:21:17,425 WARNING Null anomaly: company — 46 nulls (4.6%)
2026-04-03 01:21:17,426 WARNING Null anomaly: pickup_latitude — 1000 nulls (100.0%)
2026-04-03 01:21:17,427 WARNING Null anomaly: pickup_longitude — 1000 nulls (100.0%)
2026-04-03 01:21:17,427 WARNING Null 

=== Anomalies ===
- pickup_census_tract: 996 nulls (99.6%)
- dropoff_census_tract: 996 nulls (99.6%)
- pickup_community_area: 1000 nulls (100.0%)
- dropoff_community_area: 1000 nulls (100.0%)
- tolls: 32 nulls (3.2%)
- company: 46 nulls (4.6%)
- pickup_latitude: 1000 nulls (100.0%)
- pickup_longitude: 1000 nulls (100.0%)
- pickup_location: 1000 nulls (100.0%)
- dropoff_latitude: 1000 nulls (100.0%)
- dropoff_longitude: 1000 nulls (100.0%)
- dropoff_location: 1000 nulls (100.0%)
- trip_seconds: 702 zero values
- trip_miles: 791 zero values
- fare: 5 zero values
- tips: 283 zero values
- tolls: 967 zero values
- extras: 884 zero values
- trip_total: 5 zero values

=== Diagnosis ===
### Anomalies and Recommendations:

1. **pickup_census_tract: 996 nulls (99.6%)**
   - **Cause**: Lack of GPS data or mapping errors when translating pickup locations to census tracts.
   - **Recommendation**: Implement a verification step to assign census tracts based on pickup coordinates before ingestion. I